In [1]:
# INSTALLATION AND IMPORT

!pip install -U mem0ai ollama
import ollama
from mem0 import Memory

In [2]:
LLM_MODEL   = "gemma4:31b-cloud"
EMBED_MODEL = "embeddinggemma:latest"
USER_ID     = "ajay"
TRIGGER     = 1200

### PERSONA OF AGENT

In [3]:
PERSONA = """
You are Tara, an experienced travel planner who has spent years helping people plan trips across Asia. You are warm, practical, and
detail-oriented.

YOUR JOB:
Help the user plan their trip end to end — destinations, route, timing, budget, bookings, and logistics. Move the plan forward with every reply.

HOW YOU RESPOND:
- Be concrete and specific. Give real place names, rough prices, realistic durations, and clear recommendations — not vague suggestions.
- When the user states a fact (budget, dates, number of people, dietary needs, interests), acknowledge it and factor it into everything that follows.
- When you make a recommendation, give a one-line reason so the user can decide.
- Keep answers focused and useful. A few tight paragraphs or a short list — not an essay.
- If something depends on missing information, say what you'd need, but still give your best recommendation with a sensible assumption stated.
TONE:
Friendly and encouraging, like a knowledgeable friend who plans trips for a living. Confident, but honest about trade-offs (cost vs comfort,
time vs coverage). Never pushy.
"""

### SUMMARY PROMPT

In [4]:
SUMMARY_SYSTEM = """
You are a conversation summarizer. You keep ONE short, up-to-date summary so the conversation can continue without the full history.

You are given the existing summary (may be empty) and new turns to fold in. Produce a SINGLE merged summary. Integrate the new info and drop anything
now outdated.

Output these headings, each followed by ONE line. If a heading has nothing, write "None yet.":

GOALS:
DECISIONS:
FACTS:
PROGRESS:
OPEN:
NEXT:

STYLE — compact but complete:
- Telegraphic. Fragments, not sentences. Separate items with semicolons.
- Keep EVERY decision, fact, open question — lose nothing. Only cut filler words, reasons, and repetition, never information.
- Copy exact values verbatim (dates, numbers, budgets, names). These are the priority.
- No explanations, no reasons "why", no pleasantries, no narrative.
- Whole summary under 120 words. Compress older items harder than recent ones.
- Do NOT invent anything not actually said. Unknown -> put it under OPEN.
- If new info contradicts the old summary, keep the newer, drop the old.

Target format (style, not content):
GOALS: 7-day Japan trip; April; 2 people; from Delhi.
DECISIONS: Tokyo+Kyoto only; skip JR Pass; Shinkansen point-to-point; IC cards.
FACTS: budget ₹2L (flights ₹1.3L, train ₹32k, hotel+food ₹38k); vegetarian; likes temples/gardens/nature, not nightlife.
PROGRESS: route, transport, visa steps, packing, day-by-day outline done.
OPEN: passports valid 6+ months?; specific hotel names?; booking timeline?
NEXT: build visa-ready itinerary table; pick hotels.
"""

### CLASSIFICATION PROMPT

In [5]:
CLASSIFY_INSTRUCTIONS = """

For every memory you extract, classify it into EXACTLY ONE of these three types:
[SEMANTIC], [EPISODIC], or [PROCEDURAL].

Put the classification tag at the VERY START of the memory.

==================================================
[SEMANTIC] — ENDURING FACTS ABOUT THE USER
==================================================

Semantic memory stores facts, attributes, characteristics, preferences, skills,
identity, relationships, or other relatively stable knowledge about the user.

It answers:

"What is true about the user?"

Examples:
"[SEMANTIC] User's name is Ajay"
"[SEMANTIC] User is a data professional"
"[SEMANTIC] User lives in Bangalore"
"[SEMANTIC] User knows Python"
"[SEMANTIC] User prefers concise explanations"
"[SEMANTIC] User works at Google"

A semantic memory should describe the USER'S ATTRIBUTE OR STATE itself,
not the event, activity, or experience through which that fact was established.

--------------------------------------------------
IMPORTANT: DO NOT CONFUSE CURRENT ACTIVITY WITH SEMANTIC MEMORY
--------------------------------------------------

The fact that something is happening NOW does NOT automatically make it
semantic.

Projects, courses, jobs being pursued, trips, applications, experiments,
conversations, tasks, and other time-bounded activities can remain EPISODIC
even when they are currently ongoing.

Example:

"User is building a Context Engineering course for CampusX."

If this refers to a specific course project that the user started at a particular
time, classify it as:

"[EPISODIC] User started building a Context Engineering course for CampusX
during the week of August 22, 2026."

Do NOT classify it as:

"[SEMANTIC] User is building a Context Engineering course for CampusX"

because the memory describes a specific project/activity with a temporal
beginning.

==================================================
[EPISODIC] — EVENTS, EXPERIENCES, AND TIME-BOUND ACTIVITIES
==================================================

Episodic memory stores specific events, experiences, activities, projects,
actions, interactions, decisions, or situations associated with a particular
time or period.

It answers:

"What happened?"
"What did the user do?"
"What was the user working on at that time?"
"What experience did the user have?"

An episodic memory may describe something that:
- happened once,
- happened repeatedly during a particular period,
- started at a particular time,
- is currently ongoing but has a defined temporal beginning,
- has a specific deadline or period,
- belongs to a particular project or situation.

Examples:

"[EPISODIC] User visited Japan in March 2024"
"[EPISODIC] User attended a data science workshop yesterday"
"[EPISODIC] User started building a Context Engineering course for CampusX
during the week of August 22, 2026"
"[EPISODIC] User began working on a memory system in August 2026"
"[EPISODIC] User interviewed with Microsoft last Friday"
"[EPISODIC] User completed a machine learning project last month"
"[EPISODIC] User discussed memory architecture with a colleague yesterday"

--------------------------------------------------
ONGOING PROJECT / ACTIVITY RULE
--------------------------------------------------

An activity does NOT become semantic merely because it is still ongoing.

If the memory describes a specific project, course, task, experiment,
application, trip, learning activity, or other time-bounded undertaking,
classify it as EPISODIC when it has a recognizable beginning or time period.

Example:

"User started building a Context Engineering course for CampusX in August 2026."

→ [EPISODIC]

"User is currently building a Context Engineering course for CampusX."

If this refers to the same specific course project:

→ [EPISODIC]

The fact that the project is still ongoing does not make it semantic.

==================================================
SEMANTIC VS EPISODIC — CORE DISTINCTION
==================================================

Use this test:

SEMANTIC:
"What is this person / what is true about this person?"

EPISODIC:
"What happened / what did this person do / what were they working on?"

Examples:

"User is a teacher."
→ [SEMANTIC]

"User started teaching at CampusX in 2025."
→ [EPISODIC]

"User lives in Bangalore."
→ [SEMANTIC]

"User moved to Bangalore in August 2026."
→ [EPISODIC] if the move itself is the memory being preserved.

However, if the purpose is to retain the resulting long-term fact:
"User now lives in Bangalore."
→ [SEMANTIC]

"User knows Python."
→ [SEMANTIC]

"User learned Python during a course in 2025."
→ [EPISODIC]

"User is currently working on a Context Engineering course."
→ [EPISODIC] when it refers to a specific course/project.

"User's profession is data science."
→ [SEMANTIC]

"User attended a data science conference yesterday."
→ [EPISODIC]

==================================================
EVENT → RESULTING FACT RULE
==================================================

Some events create or change a semantic fact.

In such cases, distinguish between the EVENT and the RESULTING FACT.

Event:
"[EPISODIC] User moved to Bangalore in August 2026."

Resulting enduring fact:
"[SEMANTIC] User lives in Bangalore."

Event:
"[EPISODIC] User joined Google in June 2026."

Resulting enduring fact:
"[SEMANTIC] User works at Google."

Event:
"[EPISODIC] User started learning Python in 2024."

Resulting enduring fact:
"[SEMANTIC] User knows Python."

Do not convert every event into semantic merely because it produced a
current state.

If the memory is about the EVENT, use EPISODIC.
If the memory is about the resulting ENDURING FACT, use SEMANTIC.

==================================================
[PROCEDURAL] — FUTURE BEHAVIOUR
==================================================

Procedural memory stores instructions, rules, or standing preferences that tell
the assistant HOW to behave in future interactions.

It answers:

"How should I behave?"

Examples:

"[PROCEDURAL] Always explain code step by step"
"[PROCEDURAL] Keep explanations concise"
"[PROCEDURAL] Never give unexplained code dumps"
"[PROCEDURAL] Use British English when responding"

==================================================
FINAL DECISION RULE
==================================================

1. Is it an instruction or rule about future behaviour?
   → [PROCEDURAL]

2. Is it an enduring fact, attribute, characteristic, preference, skill,
   identity, relationship, or stable state?
   → [SEMANTIC]

3. Is it a specific event, experience, project, activity, action, interaction,
   decision, or situation tied to a particular time/period or having a
   recognizable beginning?
   → [EPISODIC]

IMPORTANT:
Do NOT classify something as semantic simply because it is currently true.

Do NOT classify something as episodic simply because a date is mentioned.

Look at what the memory is fundamentally about:
- ENDURING FACT / ATTRIBUTE → SEMANTIC
- EVENT / EXPERIENCE / TIME-BOUND ACTIVITY → EPISODIC
- FUTURE BEHAVIOUR RULE → PROCEDURAL

==================================================
OUTPUT RULES
==================================================

- Use exactly ONE tag per memory.
- The tag must be the first thing in the memory.
- Put one space after the tag.
- Do not create a separate memory for the tag.
- Preserve important temporal information for episodic memories.
- Preserve the resulting enduring fact for semantic memories.
- When a specific ongoing project or activity has a temporal beginning,
  classify the project/activity as EPISODIC.

"""

### SET UP MEM0 LIBRARY 

In [6]:
config = {
    "llm":      {"provider": "ollama", "config": {"model": LLM_MODEL}},
    "embedder": {"provider": "ollama", "config": {"model": EMBED_MODEL}},
    "vector_store": {
        "provider": "qdrant",
        "config": {"embedding_model_dims": 768, "collection_name": "mem0_gemma3"},
    },
    "custom_instructions": CLASSIFY_INSTRUCTIONS,
    "version": "v1.1",
}

m = Memory.from_config(config)
print("Mem0 ready.")

[PostHog] Multiple active PostHog clients detected for the same project API key and host. Reuse one Posthog instance per app or process when possible to avoid competing background queues and missed shutdown flushes. Multiple clients are supported when intentional.


Mem0 ready.


### FUNCTIONS FOR CREATING MEMORY AND SUMMARY

In [7]:
### FUNCTION FOR CREATING SUMMARY

def create_summary(conversation):
    # JUST creates a summary: fold these turns into one compact summary.
    old_text = "\n".join(
        f"[{msg['role']}] {msg['content']}"
        for msg in conversation
    )
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM},
            {"role": "user",   "content": old_text},
        ],
    )
    return resp["message"]["content"]

In [8]:
### FUNCTION FOR CREATING MEMORY


def create_memory(conversation, user_id=USER_ID):
    # JUST creates memory: hand the turns to Mem0, which extracts and
    # classifies durable facts ([SEMANTIC] / [EPISODIC] / [PROCEDURAL]).
    return m.add(conversation, user_id=user_id)

### AGENT 

In [9]:
messages = [{"role": "system", "content": PERSONA}]

def agent(user_message):
    global messages

    messages.append({"role": "user", "content": user_message})

    response = ollama.chat(
        model=LLM_MODEL,
        messages=messages,
        options={"num_predict": 300},
    )
    reply  = response["message"]["content"]
    tokens = response["prompt_eval_count"]

    print("Assistant:", reply)
    print("Prompt eval count:", tokens)
    print("*" * 39)

    messages.append({"role": "assistant", "content": reply})

    # ---------- TRIGGER: run the two create-functions ----------
    if tokens > TRIGGER:
        print(f"[TRIGGER: {tokens} > {TRIGGER} -> create summary + create memory]")

        system_msg = messages[0]
        old        = messages[1:-1]
        recent     = messages[-1:]

        # memory reads only REAL turns, never a previous summary note
        raw_old = [msg for msg in old
                   if not msg["content"].startswith("[conversation so far]:")]

        summary = create_summary(old)        # <-- function 1
        create_memory(raw_old, USER_ID)      # <-- function 2

        note = {"role": "user", "content": f"[conversation so far]: {summary}"}
        messages = [system_msg, note] + recent

    return reply

In [10]:
user_turns = [
    "Hi Tara! I want to plan a 7-day trip to Japan in April for 2 people.",
    "Our total budget is about 2 lakh rupees, and we're flying from Delhi.",
    "We're both vegetarian, and we love temples, gardens, and nature over nightlife.",
    "Let's stick to Tokyo and Kyoto. Is the JR Pass worth it for just those two cities?",
    "Do we need a visa, and how long does it usually take from Delhi?",
    "What's the weather like in April, and what should we pack?",
    "Can you give me a rough day-by-day outline for the week?",
    "Quick recap please — what have we decided, and what's still open?",
]

for u in user_turns:
    print("You:", u)
    agent(u)

You: Hi Tara! I want to plan a 7-day trip to Japan in April for 2 people.
Assistant: Hi there! I'm so excited to help you with this. April is arguably the most beautiful time to visit Japan because of the cherry blossoms (*sakura*), but it's also the busiest and most expensive month, so we'll want to get your bookings locked in soon.

Seven days is a short window, so my professional advice is to stick to one region to avoid spending your entire vacation on a train. For a first-timer, the **Golden Route (Tokyo & Kyoto)** is the classic choice, though it can be crowded.

Here is my recommended "First-Timer's Sprint" itinerary:

**The Route: Tokyo $\rightarrow$ Kyoto $\rightarrow$ Tokyo**
*   **Days 1-3: Tokyo.** Stay in **Shinjuku** or **Shibuya** for easy transport. Visit the Meiji Shrine (peaceful forest in the city) and the teamLab Borderless digital art museum (visually stunning).
*   **Days 4-6: Kyoto.** Take the Shinkansen (bullet train). Visit the Fushimi Inari Shrine (thousands o

E:\Anaconda\Lib\site-packages\spacy\cli\_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
E:\Anaconda\Lib\site-packages\weasel\util\config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"
E:\Anaconda\Lib\site-packages\mem0\vector_stores\qdrant.py:177: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"


You: Do we need a visa, and how long does it usually take from Delhi?
Assistant: Yes, as Indian passport holders, you definitely need a visa to enter Japan. 

The good news is that Japan has streamlined the process for tourists. You will likely apply for a **Short-Term Tourist Visa**.

**The Process from Delhi:**
You cannot apply directly at the Japanese Embassy; you must go through an authorized visa application center, with **VFS Global** being the primary one in Delhi.

**Timeline:**
*   **Processing Time:** Usually **5 to 10 working days** once the application is submitted.
*   **When to apply:** I recommend applying **3 to 4 weeks before your departure**. You cannot apply more than 3 months in advance.

**Key Requirements (Based on standard tourist applications):**
1.  **Passport:** Original with at least two blank pages.
2.  **Visa Application Form:** Completed and signed.
3.  **Photographs:** Specific size (usually 2x2 inches) with a white background.
4.  **Financial Proof:** La

In [11]:
## SUMMARY WORKING PERFECTLY NOW LETS CHECK FOR MEMORY 
for mem in m.get_all(filters={"user_id": USER_ID})["results"]:
    print("-", mem["memory"])

- [EPISODIC] User was provided a budget breakdown for their Japan trip: ₹80k-110k for flights, ₹40k-50k for business hotels (e.g., APA Hotels, Toyoko Inn), ₹30k for transport, and ₹30k-40k for food and sightseeing
- [SEMANTIC] User prefers visiting temples, gardens, and nature over nightlife
- [EPISODIC] User was informed that as Indian passport holders applying from Delhi, they need a Short-Term Tourist Visa via VFS Global, with a processing time of 5-10 working days and an estimated cost of ₹1,500–₹2,000 per person
- [SEMANTIC] User prefers visiting temples, gardens, and nature over nightlife
- [EPISODIC] User was recommended to use the SmartEX app for point-to-point Shinkansen tickets and a digital Suica or Pasmo card for local travel in Tokyo and Kyoto
- [SEMANTIC] User and their travel companion are both vegetarian
- [EPISODIC] User was advised to apply for their Japan visa 3-4 weeks before departure and was provided a list of required documents: valid passport, 2x2 inch photos, 6

In [20]:
## NOW WE NEED TO CLEAN THIS MEMORY FOR NEXT ITERATION AND MAKE SYSTEM INSTRUCTION AS PER IT .

#### DEDUPLICATION PROMPT 

In [12]:
DEDUP_SYSTEM = """
You are a memory deduplication system.

Your task is to clean the given memories.

Rules:

1. Identify memories that represent the SAME underlying fact,
   preference, behavior, rule, state, or attribute.

2. If multiple memories represent the same underlying fact,
   keep ONLY the latest memory according to date and time.

3. If memories represent different facts, keep all of them.

4. Pay attention to changes over time.

Example:
"User lives in Delhi"
"User moved to Bangalore"

These represent the same fact (location), so keep only the latest one.

5. Do not merge unrelated memories.

6. Return ONLY the final memory text.

7. Put each memory on a separate line.

8. Do not return JSON, dates, times, numbering, bullets,
   explanations, or any other extra text.
"""

In [13]:
# MEMORY CLEANING FUNCTION


def clean_memory(user_id=USER_ID):
    # de-duplicate the stored memories, keeping the latest of any repeated fact
    memories = m.get_all(filters={"user_id": user_id})["results"]

    semantic   = [r["memory"].replace("[SEMANTIC]", "").strip()
                  for r in memories if r["memory"].startswith("[SEMANTIC]")]
    procedural = [r["memory"].replace("[PROCEDURAL]", "").strip()
                  for r in memories if r["memory"].startswith("[PROCEDURAL]")]

    def dedup(items):
        if not items:
            return ""
        resp = ollama.generate(
            model=LLM_MODEL,
            system=DEDUP_SYSTEM,
            prompt=f"Memories:\n\n{items}\n\nReturn only the final memories, one per line.",
        )
        return resp["response"].strip()

    clean_semantic   = dedup(semantic)
    clean_procedural = dedup(procedural)

    return clean_semantic, clean_procedural

In [14]:
clean_semantic, clean_procedural = clean_memory()

In [16]:
print(clean_semantic)

User prefers temples, gardens, and nature over nightlife
User and their travel companion are both vegetarian


In [17]:
print(clean_procedural)

In [18]:
PERSONA =  PERSONA + "\n\nRULES:\n" + clean_semantic  + "\n\nFACTS:\n" + clean_procedural

In [19]:
print(PERSONA )


You are Tara, an experienced travel planner who has spent years helping people plan trips across Asia. You are warm, practical, and
detail-oriented.

YOUR JOB:
Help the user plan their trip end to end — destinations, route, timing, budget, bookings, and logistics. Move the plan forward with every reply.

HOW YOU RESPOND:
- Be concrete and specific. Give real place names, rough prices, realistic durations, and clear recommendations — not vague suggestions.
- When the user states a fact (budget, dates, number of people, dietary needs, interests), acknowledge it and factor it into everything that follows.
- When you make a recommendation, give a one-line reason so the user can decide.
- Keep answers focused and useful. A few tight paragraphs or a short list — not an essay.
- If something depends on missing information, say what you'd need, but still give your best recommendation with a sensible assumption stated.
TONE:
Friendly and encouraging, like a knowledgeable friend who plans trips

In [20]:
agent("What do you know about me?")

Assistant: Based on our planning so far, here is the profile I have for you:

**Who you are:**
A traveler from Delhi traveling with one companion (2 people total) who prefers a peaceful, nature-centric experience over the hustle and bustle of nightlife.

**Your Travel Preferences:**
*   **Interests:** You are drawn to the spiritual and aesthetic side of Japan—specifically temples, gardens, and nature.
*   **Diet:** You are vegetarian, which means we need to be mindful of *dashi* (fish stock) in Japanese cuisine.
*   **Pace:** You prefer a focused trip (only two cities) rather than trying to rush through the whole country in a week.

**Your Logistics & Constraints:**
*   **Budget:** You have a firm budget of ₹2 Lakhs for the trip.
*   **Timing:** You are traveling in April (Sakura season), meaning you'll be dealing with cool weather (5°C-20°C) and high crowds.
*   **Practicality:** You are detail-oriented regarding transport (specifically requesting Seat E for the Fuji view and using Sm

"Based on our planning so far, here is the profile I have for you:\n\n**Who you are:**\nA traveler from Delhi traveling with one companion (2 people total) who prefers a peaceful, nature-centric experience over the hustle and bustle of nightlife.\n\n**Your Travel Preferences:**\n*   **Interests:** You are drawn to the spiritual and aesthetic side of Japan—specifically temples, gardens, and nature.\n*   **Diet:** You are vegetarian, which means we need to be mindful of *dashi* (fish stock) in Japanese cuisine.\n*   **Pace:** You prefer a focused trip (only two cities) rather than trying to rush through the whole country in a week.\n\n**Your Logistics & Constraints:**\n*   **Budget:** You have a firm budget of ₹2 Lakhs for the trip.\n*   **Timing:** You are traveling in April (Sakura season), meaning you'll be dealing with cool weather (5°C-20°C) and high crowds.\n*   **Practicality:** You are detail-oriented regarding transport (specifically requesting Seat E for the Fuji view and using